In [1]:
from google.colab import drive
drive.mount('/content/drive')

DatasetPath = "/content/drive/MyDrive/dataset/brain-tumor-mri-dataset"
Training_data_path = f"{DatasetPath}/Training"
Testing_data_path = f"{DatasetPath}/Testing"

Mounted at /content/drive


In [2]:
import tensorflow as tf

IMG_SIZE = (256, 256)
BATCH_SIZE = 32

Train_data = tf.keras.utils.image_dataset_from_directory(
    Training_data_path, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=42
)
Test_data = tf.keras.utils.image_dataset_from_directory(
    Testing_data_path, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
)

class_names = Train_data.class_names
print(class_names)

AUTOTUNE = tf.data.AUTOTUNE
Train_data = Train_data.cache().shuffle(1000).prefetch(AUTOTUNE)
Test_data = Test_data.cache().prefetch(AUTOTUNE)

Found 5600 files belonging to 4 classes.
Found 1600 files belonging to 4 classes.
['glioma', 'meningioma', 'notumor', 'pituitary']


In [3]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="data_augmentation")

In [4]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(256, 256, 3)
)
base_model.trainable = False  # freeze for phase 1

inputs = layers.Input(shape=(256, 256, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)          # EfficientNet's own normalization — do NOT add Rescaling on top of this
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(4, activation="softmax")(x)

model_tl = Model(inputs, outputs, name="efficientnet_transfer")

model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_tl.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "efficientnet_transfer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 8, 8, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,055 (16.08 MB)

 Trainable params: 164,484 (642.52 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [5]:
checkpoint_tl = tf.keras.callbacks.ModelCheckpoint(
    filepath="best_model_transfer.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping_tl = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=6,
    mode="max",
    restore_best_weights=True,
    verbose=1
)

reduce_lr_tl = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

In [ ]:
history_phase1 = model_tl.fit(
    Train_data,
    validation_data=Test_data,
    epochs=15,
    callbacks=[checkpoint_tl, early_stopping_tl, reduce_lr_tl]
)

Epoch 1/15


In [ ]:
base_model.trainable = True

# freeze everything except the last ~30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # much lower LR — critical, don't skip this
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_phase2 = model_tl.fit(
    Train_data,
    validation_data=Test_data,
    epochs=15,
    callbacks=[checkpoint_tl, early_stopping_tl, reduce_lr_tl]
)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_true, y_pred = [], []
for images, labels in Test_data:
    preds = model_tl.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.show()